# Пример - Матроидные параметры и лесные характеристики графов

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **1487709**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/148770970a5d0005c8e16b8eb63969d2afe3ca22.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Семь расчётов проверяют независимость рёбер, лесные коэффициенты, резистивный радиус и переворот Уитни.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 1487709](https://github.com/magin67/polyform_algebra/commit/148770970a5d0005c8e16b8eb63969d2afe3ca22).


## Цикловой матроид треугольника

Переберём подмножества рёбер треугольника и проверим независимость внешним произведением. Все семь собственных подмножеств независимы, а произведение по трём рёбрам цикла равно нулю.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%86%D0%B8%D0%BA%D0%BB%D0%BE%D0%B2%D0%BE%D0%B9-%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4-%D0%B8-%D0%B2%D0%BD%D0%B5%D1%88%D0%BD%D0%B8%D0%B5-%D0%BF%D1%80%D0%BE%D0%B8%D0%B7%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D1%8F-%D1%80%D1%91%D0%B1%D0%B5%D1%80#%D0%BD%D0%B5%D0%B7%D0%B0%D0%B2%D0%B8%D1%81%D0%B8%D0%BC%D0%BE%D1%81%D1%82%D1%8C-%D0%BB%D0%B5%D1%81%D0%B0-%D0%B8-%D1%86%D0%B8%D0%BA%D0%BB%D1%8B).

In [3]:
from fractions import Fraction as Q
from functools import reduce
from itertools import combinations
from math import prod

from polyform_algebra.core import Boundary, Unit, circumsphere, contract, network, resistance, wedge
from polyform_algebra.matroid import (
    edge_forest_probabilities, forest_coefficients, forest_power_sums,
    forest_resistance_radius, forest_resistance_scales, normalized_forest_coefficients,
)
from polyform_algebra.toponomes import forest_toponome, laplacian_polynomial


def unit_network(labels, pairs):
    return network(tuple(labels), [(a, b, 1) for a, b in pairs])


def edge_product(net, pairs):
    positions = {label: i for i, label in enumerate(net.context.labels)}
    blades = (Boundary(net.context, (positions[a], positions[b])) for a, b in pairs)
    return reduce(lambda left, right: wedge(left, right).value, blades, Unit(net.context).as_exterior())


def subsets(items):
    return (part for size in range(len(items) + 1) for part in combinations(items, size))


triangle_edges = (("a", "b"), ("a", "c"), ("b", "c"))
triangle = unit_network("abc", triangle_edges)
triangle_forests = [part for part in subsets(triangle_edges) if edge_product(triangle, part).coefficients]
assert not edge_product(triangle, triangle_edges).coefficients
assert tuple(sum(len(part) == k for part in triangle_forests) for k in range(3)) == (1, 3, 3)
print("Independent edge sets:", len(triangle_forests))

Independent edge sets: 7


## Обычные и корневые леса K4

Сравним обычные коэффициенты с корневыми коэффициентами из $\det(I+tL)$. Значения обычного полинома в четырёх точках сверим со специализацией $t^3T_{K_4}(1+1/t,1)$, вычисленной по независимым подмножествам.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B---%D0%B3%D1%80%D0%B5%D0%B9%D0%B4,-%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82%D1%8B-%D0%B8-%D0%BA%D0%BE%D0%B3%D1%80%D0%B5%D0%B9%D0%B4#%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%B5%D0%B4%D0%B8%D0%BD%D0%B8%D1%87%D0%BD%D1%8B%D0%B9-k4).

In [4]:
k4_edges = tuple(combinations("abcd", 2))
k4 = unit_network("abcd", k4_edges)
ordinary = forest_coefficients(k4)
rooted = laplacian_polynomial(k4)
assert ordinary == (1, 6, 15, 16)
assert rooted == (1, 12, 48, 64)

# At y=1 only the independent subsets survive in the Tutte rank sum.
# A subset of size k then contributes (x - 1) ** (3 - k).
independent = [part for part in subsets(k4_edges) if edge_product(k4, part).coefficients]
for t in (Q(1), Q(2), Q(1, 2), Q(-1)):
    tutte_value = sum((1 / t) ** (3 - len(part)) for part in independent)
    assert sum(value * t**k for k, value in enumerate(ordinary)) == t**3 * tutte_value
print("Ordinary K4:", tuple(map(str, ordinary)))
print("Rooted K4:", tuple(map(str, rooted)))

Ordinary K4: ('1', '6', '15', '16')
Rooted K4: ('1', '12', '48', '64')


## Нормированный полином и суммы Ньютона

Для пути с проводимостями $1,2,3$ сравним суммы Ньютона со степенными суммами обратных проводимостей. Для единичного треугольника найдём первые четыре суммы без факторизации полинома.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%BD%D1%8B%D0%B9-%D0%BB%D0%B5%D1%81%D0%BD%D0%BE%D0%B9-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC-%D0%B8-%D1%81%D1%82%D0%B5%D0%BF%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B).

In [5]:
weighted_path = network(tuple("abcd"), [("a", "b", 1), ("b", "c", 2), ("c", "d", 3)])
normalized = normalized_forest_coefficients(weighted_path)
assert normalized == (1, Q(11, 6), 1, Q(1, 6))
path_powers = forest_power_sums(normalized, 4)
assert path_powers == tuple(sum(Q(1, c)**k for c in (1, 2, 3)) for k in range(1, 5))
triangle_powers = forest_power_sums(normalized_forest_coefficients(triangle), 4)
assert triangle_powers == (1, Q(1, 3), 0, Q(-1, 9))
print("Triangle Newton sums:", tuple(map(str, triangle_powers)))

Triangle Newton sums: ('1', '1/3', '0', '-1/9')


## Лесные резистивные масштабы

У пути и звезды на четырёх вершинах одинаковы обычные масштабы, а первые корневые различаются. Для цикла и полного графа также проверим точные отношения соседних коэффициентов.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B5-%D0%BC%D0%B0%D1%81%D1%88%D1%82%D0%B0%D0%B1%D1%8B#%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D1%8C%D1%8F-%D1%86%D0%B8%D0%BA%D0%BB%D1%8B-%D0%B8-%D0%BF%D0%BE%D0%BB%D0%BD%D1%8B%D0%B9-%D0%B3%D1%80%D0%B0%D1%84).

In [6]:
path = unit_network("abcd", (("a", "b"), ("b", "c"), ("c", "d")))
star = unit_network("abcd", (("a", "b"), ("a", "c"), ("a", "d")))
cycle = unit_network("abcd", (("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")))
assert forest_resistance_scales(forest_coefficients(path)) == (3, 1, Q(1, 3))
assert forest_resistance_scales(forest_coefficients(star)) == (3, 1, Q(1, 3))
assert forest_resistance_scales(forest_coefficients(cycle)) == (Q(3, 2), Q(2, 3), Q(1, 4))
assert forest_resistance_scales(laplacian_polynomial(path))[0] == Q(5, 2)
assert forest_resistance_scales(laplacian_polynomial(star))[0] == Q(9, 4)
assert forest_resistance_scales(ordinary) == (Q(15, 16), Q(2, 5), Q(1, 6))
assert forest_resistance_scales(rooted) == (Q(3, 4), Q(1, 4), Q(1, 12))
print("First rooted scales (path, star):", Q(5, 2), Q(9, 4))

First rooted scales (path, star): 5/2 9/4


## Вероятности рёбер взвешенного треугольника

Сверим $c_eR_e$ с вероятностями из явного перебора трёх остовных деревьев. Проверим сумму вероятностей и равенство обычных лесных полиномов при удалении и стягивании ребра $ab$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%80%D1%91%D0%B1%D0%B5%D1%80%D0%BD%D1%8B%D0%B5-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D1%8B-%D0%B8-%D1%81%D0%BB%D1%83%D1%87%D0%B0%D0%B9%D0%BD%D0%BE%D0%B5-%D0%BE%D1%81%D1%82%D0%BE%D0%B2%D0%BD%D0%BE%D0%B5-%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B).

In [7]:
weighted_triangle = network(tuple("abc"), [("a", "b", 1), ("a", "c", 2), ("b", "c", 3)])
trees = tuple(combinations(weighted_triangle.edges, 2))
tree_weights = {tree: prod(edge[2] for edge in tree) for tree in trees}
total = sum(tree_weights.values())
by_enumeration = {
    edge[:2]: sum(weight for tree, weight in tree_weights.items() if edge in tree) / total
    for edge in weighted_triangle.edges
}
probabilities = edge_forest_probabilities(weighted_triangle)
assert probabilities == by_enumeration
assert tuple(probabilities.values()) == (Q(5, 11), Q(8, 11), Q(9, 11))
assert sum(probabilities.values()) == 2

without_ab = network(tuple("abc"), weighted_triangle.edges[1:])
contracted_ab = contract(weighted_triangle, {"ab": ("a", "b"), "c": ("c",)}).value.network
f_delete = forest_coefficients(without_ab)
f_contract = forest_coefficients(contracted_ab)
# The conductance of ab is 1, and including ab raises the grade by one.
assert forest_coefficients(weighted_triangle) == tuple(
    a + b for a, b in zip(f_delete, (0, *f_contract), strict=True)
)
print("Weighted triangle probabilities:", tuple(map(str, probabilities.values())))

Weighted triangle probabilities: ('5/11', '8/11', '9/11')


## Радиус резистивной сферы

Для шести графов сравним точные значения $\rho=\xi/3-h_2/12$ с квадратом радиуса из расчёта описанной сферы по метрике сопротивлений.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D1%8B%D0%B9-%D1%80%D0%B0%D0%B4%D0%B8%D1%83%D1%81---%D0%BB%D0%B5%D1%81%D0%BD%D0%B0%D1%8F-%D1%84%D0%BE%D1%80%D0%BC%D1%83%D0%BB%D0%B0-%D0%B8-%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D0%B0%D1%8F-%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D0%BD%D0%BE%D1%81%D1%82%D1%8C#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D1%8B-%D0%B8-%D0%BD%D0%B5%D0%B7%D0%B0%D0%B2%D0%B8%D1%81%D0%B8%D0%BC%D0%BE%D0%B5-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5).

In [8]:
examples = {
    "P4": (path, Q(3, 4)),
    "star": (star, Q(3, 4)),
    "weighted P4": (weighted_path, Q(11, 24)),
    "C4": (cycle, Q(5, 16)),
    "K4": (k4, Q(3, 16)),
    "weighted triangle": (weighted_triangle, Q(15, 121)),
}
for name, (graph, expected) in examples.items():
    forest_radius = forest_resistance_radius(graph)
    geometric_radius = circumsphere(graph).value.resistance_radius
    assert forest_radius == geometric_radius == expected
    print(name, "radius:", forest_radius)

P4 radius: 3/4
star radius: 3/4
weighted P4 radius: 11/24
C4 radius: 5/16
K4 radius: 3/16
weighted triangle radius: 15/121


## Переворот Уитни

Сравним независимость всех 64 наборов меток рёбер до и после переворота. Обычные лесные коэффициенты и радиус сохраняются. Коэффициент лесов с блоками размеров $5$ и $1$ меняется с $11$ на $10$, а $R(p,q)$ — с $2$ на $3$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D1%8B,-%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B/%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D1%8B%D0%B5-%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%8B-%D0%B8-%D0%BB%D0%B5%D1%81%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B8-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B8-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D1%8B-%D0%BC%D0%B0%D1%82%D1%80%D0%BE%D0%B8%D0%B4%D0%BD%D0%BE%D0%B9-%D0%B8%D0%BD%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%82%D0%BD%D0%BE%D1%81%D1%82%D0%B8#%D0%B4%D0%B2%D1%83%D1%85%D1%82%D0%BE%D1%87%D0%B5%D1%87%D0%BD%D0%B0%D1%8F-%D1%81%D0%BA%D0%BB%D0%B5%D0%B9%D0%BA%D0%B0-%D0%B8-%D0%BF%D0%B5%D1%80%D0%B5%D0%B2%D0%BE%D1%80%D0%BE%D1%82-%D1%83%D0%B8%D1%82%D0%BD%D0%B8).

In [9]:
straight_edges = (("a", "x"), ("x", "b"), ("a", "p"), ("a", "y"), ("y", "b"), ("a", "q"))
twisted_edges = (("a", "x"), ("x", "b"), ("a", "p"), ("b", "y"), ("y", "a"), ("b", "q"))
straight = unit_network("abxypq", straight_edges)
twisted = unit_network("abxypq", twisted_edges)
by_edge_label = tuple(subsets(tuple("ABCDEF")))
for chosen in by_edge_label:
    indices = tuple("ABCDEF".index(label) for label in chosen)
    first = edge_product(straight, (straight_edges[i] for i in indices))
    second = edge_product(twisted, (twisted_edges[i] for i in indices))
    assert bool(first.coefficients) == bool(second.coefficients)
assert forest_coefficients(straight) == forest_coefficients(twisted) == (1, 6, 15, 20, 14, 4)
assert forest_resistance_radius(straight) == forest_resistance_radius(twisted) == Q(13, 16)
assert forest_toponome(straight)[(4,)] == 11
assert forest_toponome(twisted)[(4,)] == 10
pair_values = tuple(resistance(graph, pairs=(("p", "q"),)).value.values[0]
                    for graph in (straight, twisted))
assert pair_values == (2, 3)
print("Whitney twist:", len(by_edge_label), "edge subsets checked; R(p,q) =", *pair_values)

Whitney twist: 64 edge subsets checked; R(p,q) = 2 3
